# 11 — Orchestrator (LangGraph)

**Module notebook — definitions only.**

Wires `guardrail_input`, `orchestrator_node`, `content_agent`, `rag_agent`, and
`guardrail_output` into one `StateGraph`.

Flow: `guardrail_input` → (blocked? → `END` : → `orchestrator`) → (`content_agent`
or `rag_agent`) → `guardrail_output` → `END`.

A blocked request never reaches an agent — it goes straight from
`guardrail_input` to `END` with `state["blocked"] = True` and the reasons in
`state["errors"]`.

Depends on: `09_agents.ipynb`, `10_guardrails.ipynb` (both loaded earlier in `main.ipynb`).

In [ ]:
from typing import TypedDict, Optional, Any
from langgraph.graph import StateGraph, END


## 1. Shared state

`blocked` and `errors` are new here — every node can append to `errors`, and `blocked` is what the two conditional edges below check.

`audio_type`, `transcript_language`, and `summary_language` replace the old
single `language` field — three independent settings instead of one setting
doing three jobs (see `09_agents.ipynb`).

In [ ]:
class PipelineState(TypedDict, total=False):
    source: str
    audio_type: str              # "auto" | "hinglish" — which ASR model transcribes the audio
    transcript_language: str     # "english" | "arabic" — language of the translated transcript
    summary_language: str        # "english" | "arabic" — language of summary/extraction/PDF
    raw_transcript: str          # exactly what was said, auto-detected — see 02_transcriber.ipynb
    transcript: str              # translated transcript, in transcript_language
    title: str
    summary: str
    action_items: str
    key_decisions: str
    open_questions: str
    pdf_path: str
    llama_index_bundle: Any
    question: Optional[str]
    answer: Optional[str]
    chat_history: list
    route: Optional[str]
    errors: list
    blocked: bool


## 2. Orchestrator node

Unchanged from before — routing is still rule-based, swap for an LLM classifier later without touching the graph wiring.

In [ ]:
def orchestrator_node(state: PipelineState) -> PipelineState:
    route = "rag" if state.get("question") else "content"
    print(f"Orchestrator: routing to \'{route}\' agent.")
    return {**state, "route": route}


def _route_decision(state: PipelineState) -> str:
    return state.get("route", "content")


def _input_ok(state: PipelineState) -> str:
    return "blocked" if state.get("blocked") else "ok"


## 3. Build the graph

Two conditional edges now: one right after `guardrail_input` (short-circuit to `END` if blocked), one after `orchestrator` (pick the agent).

In [ ]:
def build_graph():
    graph = StateGraph(PipelineState)

    graph.add_node("guardrail_input", guardrail_input)
    graph.add_node("orchestrator", orchestrator_node)
    graph.add_node("content_agent", content_agent)
    graph.add_node("rag_agent", rag_agent)
    graph.add_node("guardrail_output", guardrail_output)

    graph.set_entry_point("guardrail_input")
    graph.add_conditional_edges(
        "guardrail_input",
        _input_ok,
        {"blocked": END, "ok": "orchestrator"},
    )
    graph.add_conditional_edges(
        "orchestrator",
        _route_decision,
        {"content": "content_agent", "rag": "rag_agent"},
    )
    graph.add_edge("content_agent", "guardrail_output")
    graph.add_edge("rag_agent", "guardrail_output")
    graph.add_edge("guardrail_output", END)

    return graph.compile()


pipeline_graph = build_graph()


## 4. Convenience wrappers

`run_new_meeting` now takes `audio_type`, `transcript_language`, and
`summary_language` as three separate arguments instead of one `language`.
Callers should check `result["blocked"]` before assuming keys like `summary`
or `answer` are present.

In [ ]:
def run_new_meeting(
    source: str,
    audio_type: str = "auto",
    transcript_language: str = "english",
    summary_language: str = "english",
) -> PipelineState:
    """Kick off the pipeline for a brand-new recording. Routes to content_agent
    (or short-circuits at guardrail_input if the request is invalid)."""
    initial_state: PipelineState = {
        "source": source,
        "audio_type": audio_type,
        "transcript_language": transcript_language,
        "summary_language": summary_language,
    }
    return pipeline_graph.invoke(initial_state)


def ask_meeting_question(state: PipelineState, question: str) -> PipelineState:
    """Ask a follow-up question against an already-processed meeting. Routes to
    rag_agent (or short-circuits at guardrail_input if the request is invalid)."""
    next_state = {**state, "question": question}
    return pipeline_graph.invoke(next_state)
